In [1]:
## langchain imports

# from langchain.text_splitter import RecursiveCharacterTextSplitter # old
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

d:\Learning\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Sample data

In [2]:
## create sample documents
sample_docs = [
    """Machine Learning Basics

    Machine Learning (ML) is a subset of Artificial Intelligence that enables systems to learn patterns from data and make predictions or decisions without explicit programming. It revolves around three key components — Task (T), Experience (E), and Performance (P) — where models improve performance on a task through experience over time.

    Core Types of ML include:
    Supervised Learning – Uses labeled data for tasks like classification (predicting categories) and regression (predicting continuous values).
    Unsupervised Learning – Works with unlabeled data to find patterns, such as clustering or dimensionality reduction.
    Reinforcement Learning – Learns optimal actions through trial and error to maximize rewards.
    Additional: Semi-Supervised and Self-Supervised Learning for scenarios with limited labeled data.
    """,

    """
    Deep Learning Essentials

    Deep learning is a subset of machine learning that uses artificial neural networks with multiple layers to automatically learn complex patterns from large datasets. Inspired by the human brain, these networks consist of input layers, hidden layers, and output layers, where each neuron applies nonlinear transformations to extract features and make predictions.
    Key differences from traditional machine learning include automated feature extraction, end-to-end learning, and the ability to handle unstructured data like images, audio, and text. However, deep learning requires large datasets, high computational power (GPUs/TPUs), and often suffers from interpretability challenges.
    """,

    """
    Large Language Models

    Large Language Models (LLMs) are advanced deep learning systems trained on massive datasets—often billions or trillions of words—to understand, generate, and manipulate human-like language. They are built on transformer architectures that use self-attention mechanisms to process sequences in parallel, capturing context and relationships between words even when far apart in text .
    Core Functionality: LLMs work by predicting the next token (word or subword) in a sequence based on context. This enables them to perform tasks like text generation, summarization, translation, code writing, sentiment analysis, and even multimodal outputs such as images or audio when extended into Large Multimodal Models (LMMs)
    """
]

sample_docs

['Machine Learning Basics\n\n    Machine Learning (ML) is a subset of Artificial Intelligence that enables systems to learn patterns from data and make predictions or decisions without explicit programming. It revolves around three key components — Task (T), Experience (E), and Performance (P) — where models improve performance on a task through experience over time.\n\n    Core Types of ML include:\n    Supervised Learning – Uses labeled data for tasks like classification (predicting categories) and regression (predicting continuous values).\n    Unsupervised Learning – Works with unlabeled data to find patterns, such as clustering or dimensionality reduction.\n    Reinforcement Learning – Learns optimal actions through trial and error to maximize rewards.\n    Additional: Semi-Supervised and Self-Supervised Learning for scenarios with limited labeled data.\n    ',
 '\n    Deep Learning Essentials\n\n    Deep learning is a subset of machine learning that uses artificial neural network

In [3]:
### save sample docs
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    print(f"path: {temp_dir}/doc_{i}.txt")
    with open(f"{temp_dir}/doc_{i}.txt", "w", encoding="utf-8") as f:
        f.write(doc)

print(f"Sample document create in : {temp_dir}")

path: C:\Users\NILANJ~1.PAU\AppData\Local\Temp\tmpgi6jnjye/doc_0.txt
path: C:\Users\NILANJ~1.PAU\AppData\Local\Temp\tmpgi6jnjye/doc_1.txt
path: C:\Users\NILANJ~1.PAU\AppData\Local\Temp\tmpgi6jnjye/doc_2.txt
Sample document create in : C:\Users\NILANJ~1.PAU\AppData\Local\Temp\tmpgi6jnjye


In [4]:
temp_dir

'C:\\Users\\NILANJ~1.PAU\\AppData\\Local\\Temp\\tmpgi6jnjye'

### 2. Document loader

In [5]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf'},
    show_progress=True
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")

100%|██████████| 3/3 [00:00<00:00, 583.33it/s]

Loaded 3 documents

First document preview:
Machine Learning Basics

    Machine Learning (ML) is a subset of Artificial Intelligence that enables systems to learn patterns from data and make predictions or decisions without explicit programmin...


### Document Splitting

In [6]:
## Initialize the text Splitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # Max for each chunk
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""] # Reirarchy of Separators
)

chunks = text_splitter.split_documents(documents)

In [7]:
chunks

[Document(metadata={'source': 'C:\\Users\\NILANJ~1.PAU\\AppData\\Local\\Temp\\tmpgi6jnjye\\doc_0.txt'}, page_content='Machine Learning Basics\n\n    Machine Learning (ML) is a subset of Artificial Intelligence that enables systems to learn patterns from data and make predictions or decisions without explicit programming. It revolves around three key components — Task (T), Experience (E), and Performance (P) — where models improve performance on a task through experience over time.'),
 Document(metadata={'source': 'C:\\Users\\NILANJ~1.PAU\\AppData\\Local\\Temp\\tmpgi6jnjye\\doc_0.txt'}, page_content='Core Types of ML include:\n    Supervised Learning – Uses labeled data for tasks like classification (predicting categories) and regression (predicting continuous values).\n    Unsupervised Learning – Works with unlabeled data to find patterns, such as clustering or dimensionality reduction.\n    Reinforcement Learning – Learns optimal actions through trial and error to maximize rewards.\n 

## Embedding Models

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

## Initialize a simple Embedding model (no API Key needed!)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    # model_name="sentence-transformers/paraphrase-multilingual-MinLM-L12-v2",
    multi_process=True,
    show_progress=True,
    cache_folder="./../model_cache/",
    # model_kwargs = {"device": "cpu"}
    # model_kwargs = {"device": "gpu"}
    # model_kwargs = {"device": "cuda"}
)
embedding_model

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder='./../model_cache/', model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=True, show_progress=True)

In [9]:
sample_text = "Machine Learning is facinating"
vector = embedding_model.embed_query(sample_text)
vector

[0.037038762122392654,
 -0.089223712682724,
 0.04945294186472893,
 -0.004823900293558836,
 -0.060524024069309235,
 0.0006581124616786838,
 0.04959085211157799,
 -0.01670856401324272,
 -0.07168750464916229,
 0.06141171604394913,
 -0.02796000987291336,
 0.04393816739320755,
 0.01350246649235487,
 0.015536176040768623,
 -0.10310302674770355,
 -0.023365292698144913,
 -0.0045269508846104145,
 -0.044909995049238205,
 0.012581870891153812,
 -0.03896067291498184,
 -0.04681098461151123,
 -0.005554227624088526,
 -0.007222628220915794,
 0.006900689098984003,
 0.05475478991866112,
 0.043456193059682846,
 0.012755880132317543,
 -0.010721025057137012,
 0.06398898363113403,
 -0.024143526330590248,
 -0.03364863991737366,
 0.03451896831393242,
 -0.010241327807307243,
 0.007147525437176228,
 -0.05561354383826256,
 -0.0030984908808022738,
 0.07359659671783447,
 0.06452351063489914,
 -0.0047429585829377174,
 0.007473898120224476,
 -0.019505254924297333,
 0.003111850470304489,
 -0.00032042182283475995,
 0.

## Initialize Chromadb _Vector Store_ and **Store the Chunks** in vector representation

In [10]:
## Create a chromadb vector Store

persistentdirector = "../chroma_db"

## initialize Chromadb with HuggingFace embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory = persistentdirector,
    collection_name = "rag_collection"
)

print(f"Vector store created with {vectorstore._collection.count()} vector")
print(f"Persisted to: {persistentdirector}")

Vector store created with 16 vector
Persisted to: ../chroma_db


### Test Similarity Search

In [11]:
query ="What are the types of machine learning?"

# similar_docs = vectorstore.similarity_search(query, k=3)
similar_docs

NameError: name 'similar_docs' is not defined

## Test Similarity with Scores 

In [30]:
result_score = vectorstore.similarity_search_with_score(query, k=2)
result_score

[(Document(metadata={'source': 'C:\\Users\\NILANJ~1.PAU\\AppData\\Local\\Temp\\tmpdgdn75iy\\doc_0.txt'}, page_content='Core Types of ML include:\n    Supervised Learning – Uses labeled data for tasks like classification (predicting categories) and regression (predicting continuous values).\n    Unsupervised Learning – Works with unlabeled data to find patterns, such as clustering or dimensionality reduction.\n    Reinforcement Learning – Learns optimal actions through trial and error to maximize rewards.\n    Additional: Semi-Supervised and Self-Supervised Learning for scenarios with limited labeled data.'),
  0.5967955589294434),
 (Document(metadata={'source': 'C:\\Users\\NILANJ~1.PAU\\AppData\\Local\\Temp\\tmpgi6jnjye\\doc_0.txt'}, page_content='Core Types of ML include:\n    Supervised Learning – Uses labeled data for tasks like classification (predicting categories) and regression (predicting continuous values).\n    Unsupervised Learning – Works with unlabeled data to find pattern

## Augmenting the RAG with LLM.

In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    temperature=0.2,
    max_tokens=500
)


In [15]:
test_resp = llm.invoke("What is Retrival Augmented Generation (RAG) and how was it being used before discovering the usecase with LLM?")
test_resp

AIMessage(content='Retrieval Augmented Generation (RAG) is a model that combines the strengths of retrieval-based and generation-based approaches in natural language processing. It uses a retriever to search for relevant information from a large corpus of text and then generates a response based on that information.\n\nBefore discovering the use case with Large Language Models (LLM), RAG was primarily used in question-answering systems and information retrieval tasks. It was used to improve the accuracy and relevance of responses by incorporating information from external sources. RAG was also used in chatbots and virtual assistants to provide more informative and contextually relevant responses to user queries.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 34, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0

In [29]:
test_resp.response_metadata
"""
{'token_usage': {'completion_tokens': 127,
  'prompt_tokens': 34,
  'total_tokens': 161,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-3.5-turbo-0125',
 'system_fingerprint': None,
 'id': 'chatcmpl-DMInxY4Hl7fO9lajlDnAGPffiSdnW',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}
"""
test_resp.response_metadata['token_usage']['total_tokens']

161

## Initialize LLM, RAG Chain, Prompt Template, Query the RAG System.

### Modern RAG Chain

In [34]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [35]:
## convert vector Store to retriever

retriever = vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000232BC7CDD30>, search_kwargs={})

In [37]:
## Create prompt template
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to aswer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to aswer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [38]:
## Create document chain
document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to aswer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': Fa

In [39]:
## Create The Final RAG Chain

rag_chain = create_retrieval_chain(retriever, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000232BC7CDD30>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to aswer the question.\nIf you do

In [40]:
response = rag_chain.invoke({"input": "What are LLMs?"})

In [41]:
response

{'input': 'What are LLMs?',
 'context': [Document(metadata={'source': 'C:\\Users\\NILANJ~1.PAU\\AppData\\Local\\Temp\\tmpdgdn75iy\\doc_2.txt'}, page_content='Core Functionality: LLMs work by predicting the next token (word or subword) in a sequence based on context. This enables them to perform tasks like text generation, summarization, translation, code writing, sentiment analysis, and even multimodal outputs such as images or audio when extended into Large Multimodal Models (LMMs)'),
  Document(metadata={'source': 'C:\\Users\\NILANJ~1.PAU\\AppData\\Local\\Temp\\tmpgi6jnjye\\doc_2.txt'}, page_content='Core Functionality: LLMs work by predicting the next token (word or subword) in a sequence based on context. This enables them to perform tasks like text generation, summarization, translation, code writing, sentiment analysis, and even multimodal outputs such as images or audio when extended into Large Multimodal Models (LMMs)'),
  Document(metadata={'source': 'C:\\Users\\NILANJ~1.PAU\\

## Add new Documents to Existing Vector Store

In [42]:
vectorstore

In [47]:
new_document = """Reinforcement Learning

Reinforcement Learning is a self-learning algorithm that includes Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGO), robotics, and autonomous systems.
"""

In [48]:
new_doc = Document(
    page_content=new_document,
    metadata={"source": "mannual_addition", "topic":"reinforcement_Learning"}
)

In [49]:
new_doc

Document(metadata={'source': 'mannual_addition', 'topic': 'reinforcement_Learning'}, page_content='Reinforcement Learning\n\nReinforcement Learning is a self-learning algorithm that includes Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGO), robotics, and autonomous systems.\n')

In [52]:
## Add new docs to vector store

### Split the docs
new_chunks = text_splitter.split_documents([new_doc])

vectorstore.add_documents(new_chunks)
new_chunks, vectorstore

([Document(metadata={'source': 'mannual_addition', 'topic': 'reinforcement_Learning'}, page_content='Reinforcement Learning\n\nReinforcement Learning is a self-learning algorithm that includes Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGO), robotics, and autonomous systems.')],
 <langchain_community.vectorstores.chroma.Chroma at 0x232bc7cdd30>)

In [54]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

Added 1 new chunks to the vector store
Total vectors now: 17


## Advanced RAG Technique - Conversational Memory

In [56]:
from langchain_classic.chains import create_history_aware_retriever # Makes the retriever understand the conversational context
from langchain_core.prompts import MessagesPlaceholder # placeholder for chat history in prompts
from langchain_core.messages import AIMessage, HumanMessage # Structuring the message

In [57]:
## create prompt that includes the chat history
contextualized_q_system_prompt = """Given a chat hostory and the latest user question
which might refrence context in the chat history, formulate a standalone question 
whaich can be understood without the chat history. DO NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""


contextualized_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualized_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [58]:
## history aware retirval
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualized_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000232BC7CDD30>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag

In [59]:
# create new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved contex to answer the question.
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# create conversational RAG pipeline
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)

print("Conversation RAG Chain created")


Conversation RAG Chain created


In [61]:
chat_history = []
# First question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print("Q. What is machine learning?")
print(f"A. {result1['answer']}")

Q. What is machine learning?
A. Machine Learning is a subset of Artificial Intelligence that enables systems to learn patterns from data and make predictions or decisions without explicit programming. It revolves around three key components — Task (T), Experience (E), and Performance (P) — where models improve performance on a task through experience over time.


In [62]:
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"
})
print("Q. What are its main types?")
print(f"A. {result1['answer']}")

Q. What are its main types?
A. Machine Learning is a subset of Artificial Intelligence that enables systems to learn patterns from data and make predictions or decisions without explicit programming. It revolves around three key components — Task (T), Experience (E), and Performance (P) — where models improve performance on a task through experience over time.


In [63]:
result2

{'chat_history': [],
 'input': 'What are its main types?',
 'context': [Document(metadata={'source': 'C:\\Users\\NILANJ~1.PAU\\AppData\\Local\\Temp\\tmpdgdn75iy\\doc_0.txt'}, page_content='Core Types of ML include:\n    Supervised Learning – Uses labeled data for tasks like classification (predicting categories) and regression (predicting continuous values).\n    Unsupervised Learning – Works with unlabeled data to find patterns, such as clustering or dimensionality reduction.\n    Reinforcement Learning – Learns optimal actions through trial and error to maximize rewards.\n    Additional: Semi-Supervised and Self-Supervised Learning for scenarios with limited labeled data.'),
  Document(metadata={'source': 'C:\\Users\\NILANJ~1.PAU\\AppData\\Local\\Temp\\tmpgi6jnjye\\doc_0.txt'}, page_content='Core Types of ML include:\n    Supervised Learning – Uses labeled data for tasks like classification (predicting categories) and regression (predicting continuous values).\n    Unsupervised Learn